# Airfield data extraction 

Data provided by VTTI:
- Need to extract frames from video feed
- For colmap usage

In [1]:
import cv2
import os
import numpy as np



# SAVE YOUR WORK
%load_ext autoreload
%autoreload 2
%autosave 180

Autosaving every 180 seconds


In [2]:
def extract_frames(video_path, start_time, end_time, output_dir, target_fps, save_npy=True):
    """
    Extract image frames from a video
    Inputs: video path, start time, end time, output path
    Output: saved video in output directory
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not open video.")
        return

    # Get frames per sec
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print("frames per second: ", fps)
    print("\ntotal frames: ", total_frames)

    # Convert start and end times to frame numbers
    start_frame = int(start_time * fps)
    end_frame = int(end_time * fps)

    if end_frame > total_frames:
        end_frame = total_frames

    # Output directory 
    os.makedirs(output_dir, exist_ok=True)

    # start frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frame_num = start_frame 
    num_frames = 0

    step = int(round(fps/target_fps))

    timestamps = []
    
    while frame_num <= end_frame:
        ret, frame = cap.read()
        if not ret:
            break

        # Save
        if (frame_num-start_frame) % step == 0:
            frame_filename = os.path.join(output_dir, f'frame_{frame_num}.jpg')
            cv2.imwrite(frame_filename, frame) # IMAGE DOWNLOAD STEP 

            # COMPUTE TIMESTAMP 
            timestamp = frame_num/fps
            timestamps.append(timestamp)

            num_frames += 1
        frame_num += 1

    cap.release()
    print(f"{num_frames} frames saved to {output_dir}")

    timestamps = np.array(timestamps, dtype=np.float64)

    if save_npy:
        npy_path = os.path.join(output_dir, 'timestamps.npy')
        np.save(npy_path, timestamps)
        print(f"Timestamps saved to {npy_path}")

    return timestamps

In [3]:
path = '/home/daniel-choate/Datasets/VTTI_airfield/MSNV_0000_0000_0_250804_1830_00156/MSNV_0000_0000_0_250804_1830_00156_camera_forward.mp4'
start = 476 #430 # 7 min 10 sec
end = 479.9 #450 # 7 min 30 sec 
output_dir = '/home/daniel-choate/Datasets/VTTI_airfield/MSNV_0000_0000_0_250804_1830_00156/6_fps/Images_756_800'
target_fps = 6

extract_frames(path, start, end, output_dir, target_fps)

frames per second:  59.99851184939916

total frames:  32254
24 frames saved to /home/daniel-choate/Datasets/VTTI_airfield/MSNV_0000_0000_0_250804_1830_00156/6_fps/Images_756_800
Timestamps saved to /home/daniel-choate/Datasets/VTTI_airfield/MSNV_0000_0000_0_250804_1830_00156/6_fps/Images_756_800/timestamps.npy


array([475.99513921, 476.16181001, 476.32848081, 476.49515161,
       476.66182241, 476.82849321, 476.99516401, 477.16183481,
       477.32850561, 477.49517641, 477.66184721, 477.82851801,
       477.99518881, 478.16185961, 478.32853041, 478.49520122,
       478.66187202, 478.82854282, 478.99521362, 479.16188442,
       479.32855522, 479.49522602, 479.66189682, 479.82856762])

In [ ]:
# import cv2
# import glob
# import os

# # Input and output folders
# in_dir = "/home/daniel-choate/Datasets/VTTI_airfield/MSNV_0000_0000_0_250804_1830_00156/Images"   # folder with your raw RGB timestamped images
# out_dir = "/home/daniel-choate/Datasets/VTTI_airfield/MSNV_0000_0000_0_250804_1830_00156/Images_PP" # where processed images go
# os.makedirs(out_dir, exist_ok=True)

# # Collect and sort input files
# files = sorted(glob.glob(os.path.join(in_dir, "*")))

# for i, f in enumerate(files):
#     img = cv2.imread(f)
#     h, w = img.shape[:2]

#     # --- STEP 1: crop out timestamp ---
#     # Example: remove bottom 40 pixels (adjust for your case)
#     cropped = img[100:h, 0:w]

#     # --- STEP 2: convert to grayscale ---
#     gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)

#     # --- STEP 3: rename sequentially ---
#     out_path = os.path.join(out_dir, f"{i:06d}.png")
#     cv2.imwrite(out_path, gray)

# print(f"Processed {len(files)} images into {out_dir}")


### GPS data extraction

In [1]:
# Extract odom and bestutm
# Odom is most directly comparable

from mcap_ros2.reader import read_ros2_messages
import numpy as np

fn = '/home/daniel-choate/Datasets/VTTI_airfield/MSNV_0000_0000_0_250804_1830_00156/MSNV_0000_0000_0_250804_1830_00156-gps_0.mcap'

odom_traj = []

with open(fn, "rb") as f:
    for msg in read_ros2_messages(f, topics=["/novatel/oem7/odom"]):
        odom_msg = msg.ros_msg
        x = odom_msg.pose.pose.position.x
        y = odom_msg.pose.pose.position.y
        z = odom_msg.pose.pose.position.z
        t = msg.publish_time  # nanoseconds
        odom_traj.append([t, x, y, z,
                         odom_msg.pose.pose.orientation.x,
                          odom_msg.pose.pose.orientation.y,
                          odom_msg.pose.pose.orientation.z,
                          odom_msg.pose.pose.orientation.w])

odom_traj = np.array(odom_traj)
print("Odom trajectory shape:", odom_traj.shape)


bestutm_traj = []

with open(fn, "rb") as f:
    for msg in read_ros2_messages(f, topics=["/novatel/oem7/bestutm"]):
        bestutm_msg = msg.ros_msg
        e = bestutm_msg.easting
        n = bestutm_msg.northing
        h = bestutm_msg.height
        t = msg.publish_time
        bestutm_traj.append([t, e, n, h])

bestutm_traj = np.array(bestutm_traj)
print("BestUTM trajectory shape:", bestutm_traj.shape)

Odom trajectory shape: (53966, 8)
BestUTM trajectory shape: (540, 4)


In [2]:
output_dir = '/home/daniel-choate/Datasets/VTTI_airfield/MSNV_0000_0000_0_250804_1830_00156/odom_traj.npy'
np.save(output_dir, odom_traj)